# reaction-diffusion system
$$
\begin{cases}
u_t = 0.1\Delta u + u - u^3 - uv^2 + u^2 v + v^3, \\
v_t = 0.1\Delta v + v - u^3 - uv^2 - u^2 v - v^3,
\end{cases}
$$

In [1]:
import sys
import os
sys.path.append(os.path.abspath(r'...'))
import scikit_tt as scikit
import numpy as np
import scipy.io as sio
import tensor_auxiliary as aux

In [2]:
data = sio.loadmat(r'...\rf2d_data.mat')
t = data['t'][0]
x = data['x'][0]
y = data['y'][0]
u = data['u']
v = data['v']


# n = len(x) # also the length of y
# steps = len(t)
dx = x[2]-x[1]
dy = y[2]-y[1]
dt = (t[2]-t[1])
Nt = len(t)



In [3]:
ut = aux.dudt2(u,dt)
print(ut.shape)
uxx = np.zeros_like(u)
for i in range(Nt):
    uxx[i,:,:] = aux.diff_periodic(u[i,:,:],dx,axis=0,d=2)
uyy = np.zeros_like(u)
for i in range(Nt):
    uyy[i,:,:] = aux.diff_periodic(u[i,:,:],dx,axis=1,d=2)

vt = aux.dudt2(v,dt)
print(ut.shape)
vxx = np.zeros_like(v)
for i in range(Nt):
    vxx[i,:,:] = aux.diff_periodic(v[i,:,:],dx,axis=0,d=2)
vyy = np.zeros_like(v)
for i in range(Nt):
    vyy[i,:,:] = aux.diff_periodic(v[i,:,:],dx,axis=1,d=2)

(201, 256, 256)
(201, 256, 256)


In [4]:
# compute domain
T1 = 0
# T1 = 10
T2 = 50
X1 = 64
X2 = 128
Y1 = 64
Y2 = 128
# choose sample
np.random.seed(42)
nt = 10
nxny = 200
TT = np.linspace(T1, T2, nt+1)[:-1].astype(int)
XX = np.random.randint(X1, X2, nxny)
YY = np.random.randint(Y1, Y2, nxny)
points = np.column_stack((x, y))

XX = np.tile(XX, nt)
YY = np.tile(YY, nt)
TT = np.tile(TT, nxny)


Compute $u$

In [5]:
U = np.array([u[TT,XX,YY].reshape(nt*nxny),
              v[TT,XX,YY].reshape(nt*nxny),
             uxx[TT,XX,YY].reshape(nt*nxny)+
              uyy[TT,XX,YY].reshape(nt*nxny),
              vxx[TT,XX,YY].reshape(nt*nxny)+
              vyy[TT,XX,YY].reshape(nt*nxny)])
V = np.array([ut[TT,XX,YY].reshape(nt*nxny)])
P = [lambda t: 1, lambda t: t ,lambda t:t**2,lambda t:t**3]
print(U.shape)
print(V.shape)

(4, 2000)
(1, 2000)


In [6]:
p = len(P)
core_type_1 = np.zeros([1, p, 1, 1])
core_type_1[0, 0, 0, 0] = 1
core_type_2 = np.zeros([1, p, 1, 1])
core_type_2[0, 1, 0, 0] = 1
core_type_3 = np.zeros([1, p, 1, 1])
core_type_3[0, 2, 0, 0] = 1
core_type_4 = np.zeros([1, p, 1, 1])
core_type_4[0, 3, 0, 0] = 1
core_type_6 = np.zeros([1, 1, 1, 1])
core_type_6[0, 0, 0, 0] = 1
cores = [core_type_1]
cores.append(core_type_1)
cores.append(core_type_2)
cores.append(core_type_1)
cores.append(0.1*core_type_6)
coefficient_tensor = scikit.TT(cores) # uxx+uyy
cores = [core_type_2]
cores.append(core_type_1)
cores.append(core_type_1)
cores.append(core_type_1)
cores.append(1*core_type_6)
coefficient_tensor += scikit.TT(cores) # u
cores = [core_type_4]
cores.append(core_type_1)
cores.append(core_type_1)
cores.append(core_type_1)
cores.append(-1*core_type_6)
coefficient_tensor += scikit.TT(cores) #u3
cores = [core_type_2]
cores.append(core_type_3)
cores.append(core_type_1)
cores.append(core_type_1)
cores.append(-1*core_type_6)
coefficient_tensor += scikit.TT(cores) #v2u
cores = [core_type_3]
cores.append(core_type_2)
cores.append(core_type_1)
cores.append(core_type_1)
cores.append(1*core_type_6)
coefficient_tensor += scikit.TT(cores) #u2v
cores = [core_type_1]
cores.append(core_type_4)
cores.append(core_type_1)
cores.append(core_type_1)
cores.append(1*core_type_6)
coefficient_tensor += scikit.TT(cores) #v3
xi_exact = coefficient_tensor
xi_exact_num = xi_exact.full().flatten()
# print(xi_exact_num)

In [7]:
xi = aux.mandy_cm(U, V, P, threshold=1e-8)
xi_num1 = xi.full().flatten()
xi_formatted = [f"{x:.4f}" for x in xi_num1]
print(", ".join(xi_formatted))

-0.2030, -0.5247, -7.2770, 15.3191, -0.3476, -1.1853, -17.4825, 18.6256, -4.3822, 8.7988, 5.0493, -18.4238, -5.3460, 34.5684, -24.2994, 7.7342, -3.2897, -19.4341, 41.0352, 5.7159, -2.3378, -28.8135, 11.5998, 5.2881, -13.1681, 35.2738, 1.0956, -5.9488, 7.3871, 20.6757, -3.3751, 4.5899, -25.8695, 30.3875, 36.9758, -16.0196, -2.2640, -33.7552, -4.8647, 18.1465, 1.6803, 12.4406, 32.5007, 9.1795, -12.8946, 12.4280, -2.6808, 17.3802, -24.5877, 59.8507, -21.7980, -4.2731, 2.9146, -17.2724, -31.3144, 39.2158, 8.3289, -3.5711, 21.9891, -1.8086, -21.5049, 18.0396, -4.3550, 6.4671, 1.4560, -14.2657, 1.1394, 17.3185, -6.3983, -12.3392, 23.8751, -18.5516, -11.6250, 11.9069, -3.3318, 59.6333, -7.9265, -27.2730, -19.5414, 27.6076, -6.8834, -45.2404, 51.2470, -3.6666, -29.9376, 2.9438, 9.4689, -16.3045, -9.0875, 38.0143, 14.8127, -12.4290, -41.6355, -17.9993, -18.5055, 6.3486, -0.1015, -49.4271, -2.9285, 30.5138, -21.1179, -23.6949, -4.4111, 22.1887, 20.3887, -8.6206, -10.0499, -21.8822, -18.1350, 14.

In [8]:
iter=500
d, m = U.shape
p = len(P)
n = p ** d
b0=range(n)
e=1e-5
for i in range(iter):
    psi=aux.build_psi(U,P,lam=1e-2,beta=b0,eps=1e-5)
    xi=aux.coefficient_solving(U,psi,P,V,1e-8)
    b1=xi.full().flatten()
    if abs(b1-b0).all()<e:
        break
    b0=b1
print(i)
xi_num2 = xi.full().flatten()

xi_formatted = [f"{x:.4f}" for x in xi_num2]
print(", ".join(xi_formatted))

499
0.0004, -0.0000, 0.0000, 0.0000, 0.0989, 0.0006, 0.0000, 0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, -0.0000, -0.0002, -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, 0.0000, 0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0013, -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, 0.0000, 0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.9982, 0.0000, 0.0000, 0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, 0.0000, 0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 0.9977, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, -0.0000, 0.0000, 0.0000, -0.0000, 0.0000, -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, 0.0000, 0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, -0.0000, -0.9951, -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, 0.0000, 0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, 0.0000, 0.0016, 0.0000, 0.0000, 0.0000, -0.0000, -0.0

In [9]:
candidates = []
for u in ['', 'u','u2','u3']:
    for v in ['', 'v','v2','v3']:
        for uxx in ['', '(uxx+uyy)','(uxx+uyy)2','(uxx+uyy)3']:
            for uyy in ['', '(vxx+vyy)','(vxx+vyy)2','(vxx+vyy)3']:
                candidates.append(u+v+uxx+uyy)
candidates[0]='1'

In [10]:
rel_errors = np.linalg.norm(xi_num1 - xi_exact_num) / np.linalg.norm(xi_exact_num)
print("OLS",rel_errors)
rel_errors = np.linalg.norm(xi_num2 - xi_exact_num) / np.linalg.norm(xi_exact_num)
print("IRLS",rel_errors)

OLS 153.72875508229652
IRLS 0.003117257120136844


In [11]:
idx = [i for i,val in enumerate(xi_exact_num) if val != 0]
print(idx)
res = [f"{xi_num2[i]}{candidates[i]}" for i in idx]
print(res)

[4, 48, 64, 96, 144, 192]
['0.0989251876753011(uxx+uyy)', '0.998171617352467v3', '0.9977063553711901u', '-0.9951401248269085uv2', '0.9977428711706894u2v', '-0.9980800554412373u3']


Compute $v$

In [12]:
V = np.array([vt[TT,XX,YY].reshape(nt*nxny)])

In [13]:
p = len(P)

core_type_1 = np.zeros([1, p, 1, 1])
core_type_1[0, 0, 0, 0] = 1

core_type_2 = np.zeros([1, p, 1, 1])
core_type_2[0, 1, 0, 0] = 1

core_type_3 = np.zeros([1, p, 1, 1])
core_type_3[0, 2, 0, 0] = 1

core_type_4 = np.zeros([1, p, 1, 1])
core_type_4[0, 3, 0, 0] = 1

core_type_6 = np.zeros([1, 1, 1, 1])
core_type_6[0, 0, 0, 0] = 1

cores = [core_type_1]
cores.append(core_type_1)
cores.append(core_type_1)
cores.append(core_type_2)
cores.append(0.1*core_type_6)
coefficient_tensor = scikit.TT(cores) # vxx+vyy

cores = [core_type_1]
cores.append(core_type_2)
cores.append(core_type_1)
cores.append(core_type_1)
cores.append(1*core_type_6)
coefficient_tensor += scikit.TT(cores) # v

cores = [core_type_4]
cores.append(core_type_1)
cores.append(core_type_1)
cores.append(core_type_1)
cores.append(-1*core_type_6)
coefficient_tensor += scikit.TT(cores) #u3

cores = [core_type_2]
cores.append(core_type_3)
cores.append(core_type_1)
cores.append(core_type_1)
cores.append(-1*core_type_6)
coefficient_tensor += scikit.TT(cores) #v2u

cores = [core_type_3]
cores.append(core_type_2)
cores.append(core_type_1)
cores.append(core_type_1)
cores.append(-1*core_type_6)
coefficient_tensor += scikit.TT(cores) #u2v

cores = [core_type_1]
cores.append(core_type_4)
cores.append(core_type_1)
cores.append(core_type_1)
cores.append(-1*core_type_6)
coefficient_tensor += scikit.TT(cores) #v3

xi_exact = coefficient_tensor
xi_exact_num = xi_exact.full().flatten()
# print(xi_exact_num)


In [14]:
xi = aux.mandy_cm(U, V, P, threshold=1e-8)
xi_num1 = xi.full().flatten()
xi_array=xi.full().flatten()
print("x shape",np.array(xi_array).shape)
xi_formatted = [f"{x:.4f}" for x in xi_array]
print(", ".join(xi_formatted))

x shape (256,)
0.0687, -1.6114, 8.9937, -9.2477, 0.9774, -0.7030, 9.8811, -11.3419, 0.6417, 4.2318, -9.0830, 7.6132, -1.2502, -2.6073, 5.9437, -1.1383, -0.3820, 9.1487, 1.1719, -14.4887, 5.1632, 4.2776, -1.4465, -10.1608, 9.1662, -11.7305, 1.1130, 7.4460, -1.7651, -1.9531, -3.8662, 13.2775, 0.5920, 12.3099, -12.0238, -6.4453, 6.4056, -0.6976, -3.6529, -5.0518, 5.2160, 5.3111, -2.1660, -8.5202, 3.3731, 2.8974, -7.7427, -1.0840, 0.2341, 5.1276, -9.3274, 1.0493, -0.0400, -1.6143, 6.7348, -7.5775, 3.9376, 4.9060, 3.6635, -15.2973, 1.3843, 2.7848, -2.0204, -7.2880, 0.6186, 4.4894, -5.7319, 3.2957, -2.5269, 4.1981, -7.8879, 2.4599, -3.7293, -1.1280, 7.4215, -12.3727, 6.9710, 2.2890, -7.1458, 4.8502, 8.5881, -6.0459, 6.7415, -3.8363, -4.1013, -1.3003, -2.8065, -3.4830, -7.0127, -5.5781, 6.9978, 7.7510, 18.7257, 6.8542, -7.8849, -3.7072, 5.7120, 5.8486, -9.6474, -0.5643, 1.7734, -0.2458, -5.1667, -1.2767, 2.2394, -4.8329, 0.1765, -0.5622, 2.5885, 7.4592, 5.8801, 7.0321, 2.7054, -1.0133, 2.7705

In [15]:
iter=500
d, m = U.shape  
p = len(P) 
n = p ** d  
#b0 = xi_num1
b0=range(n)
e=1e-5
for i in range(iter):
    psi=aux.build_psi(U,P,lam=5e-3,beta=b0,eps=1e-5)
    xi=aux.coefficient_solving(U,psi,P,V,1e-8)
    b1=xi.full().flatten()
    if abs(b1-b0).all()<e:
        break
    b0=b1
print(i)
xi_num2 = xi.full().flatten()
xi_formatted = [f"{x:.4f}" for x in xi_num2]
print(", ".join(xi_formatted))

499
0.0000, 0.0967, 0.0017, 0.0000, -0.0001, -0.0000, -0.0000, 0.0000, 0.0001, 0.0000, 0.0000, -0.0000, 0.0000, -0.0000, 0.0000, 0.0000, 0.9966, -0.0000, -0.0000, 0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0008, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, -0.0000, -0.0000, 0.0000, 0.0000, 0.0000, 0.0000, -0.9963, -0.0009, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 0.0000, 0.0004, -0.0000, -0.0000, -0.0001, -0.0009, -0.0000, 0.0000, -0.0001, 0.0000, -0.0000, -0.0000, 0.0000, -0.0000, -0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, -0.0000, -0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, -0.0000, -0.0000, -0.9985, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0

In [16]:
rel_errors = np.linalg.norm(xi_num1 - xi_exact_num) / np.linalg.norm(xi_exact_num)
print("OLS",rel_errors)
rel_errors = np.linalg.norm(xi_num2 - xi_exact_num) / np.linalg.norm(xi_exact_num)
print("IRLS",rel_errors)

OLS 42.67926114828638
IRLS 0.0036735251911044686


In [17]:
candidates = []
for u in ['', 'u','u2','u3']:
    for v in ['', 'v','v2','v3']:
        for uxx in ['', '(uxx+uyy)','(uxx+uyy)2','(uxx+uyy)3']:
            for uyy in ['', '(vxx+vyy)','(vxx+vyy)2','(vxx+vyy)3']:
                candidates.append(u+v+uxx+uyy)
candidates[0]='1'

In [18]:
idx = [i for i,val in enumerate(xi_exact_num) if val != 0]
print(idx)
res = [f"{xi_num2[i]}{candidates[i]}" for i in idx]
print(res)

[1, 16, 48, 96, 144, 192]
['0.09669985086095557(vxx+vyy)', '0.9966053225768366v', '-0.9962791327904854v3', '-0.9985387252208405uv2', '-0.9954282735372464u2v', '-0.9990148929894105u3']
